# Imports and Setup


In [172]:

import pandas as pd
import numpy as np
from sodapy import Socrata

APP_KEY = "t2GPRE1GXlktxvqOIFMelPIvT"

version = "3"

client = Socrata(
    "data.cityofnewyork.us",
    APP_KEY,
    timeout=90
)

RELEASE_NAMES = ["prelim", "exec", "adopt"]

# Expense Budget


https://data.cityofnewyork.us/api/v3/views/mwzb-yiwb/query.json


In [173]:
EXPENSE_DATASET_ID = "mwzb-yiwb"

### Check Fiscal Year


In [174]:
# 1. Fetch the 2 most recent unique fiscal years present in the dataset
year_query = "SELECT fiscal_year GROUP BY fiscal_year ORDER BY fiscal_year DESC LIMIT 2"
current_fy_json = client.get(EXPENSE_DATASET_ID, query=year_query)

In [175]:
# Extract the years into a list, e.g., ['2026', '2025']
planned_fy = [row['fiscal_year'] for row in current_fy_json][0]

planned_fy_name = f"FY{int(planned_fy) % 100}"

print(planned_fy)
print(planned_fy_name)

print()

current_fy = int(planned_fy) - 1
current_fy_name = f"FY{int(current_fy) % 100}"

print(current_fy)
print(current_fy_name)

2027
FY27

2026
FY26


In [176]:
where_string = f"fiscal_year IN ({planned_fy})"

print(f"where_string:\n{where_string}\n")

orderby_arr = [
    "agency_number", # 2: 
    "unit_appropriation_number", # 4: 
    "responsibility_center_code", # 23: 
    "budget_code_number", # 6: 
    "object_class_number", # 8: 
    "object_code", # 10: 
    "publication_date DESC"
]

orderby_string = ", ".join(orderby_arr)

print(f"orderby_string:\n{orderby_string}\n")

where_string:
fiscal_year IN (2027)

orderby_string:
agency_number, unit_appropriation_number, responsibility_center_code, budget_code_number, object_class_number, object_code, publication_date DESC



### Query Data if necessary


In [177]:
data = []
offset = 0

limit = 1000

try:
    df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")
except:
    print(f"Querying for fiscal year {planned_fy}_v{version}:\n")
    while True:
        print(f"Fetching rows starting at offset: {offset}")

        temp = client.get(
            dataset_identifier=EXPENSE_DATASET_ID,
            # select=select_string,
            where=where_string,
            # group=groupby_string,
            order=orderby_string,
            # order=":id",
            limit=limit,
            offset=offset
        )
        
        if not temp:
            break
        
        # print(temp[0])
        
        data.extend(temp)
        
        offset += limit
        
        # break
            
    print(f"End of query. Successfully fetched {len(data)} total rows.")
    
    len(data)
    
    # Convert to pandas DataFrame
    df = pd.DataFrame.from_records(data)
    # df.drop(columns=["financial_plan_savings_flag"], inplace=True)
    
    df.to_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv", index=False)
    df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")

/var/folders/kq/xpb5g0qn23g4s4hvlvhwdgn80000gn/T/ipykernel_31034/944054463.py:7: DtypeWarning: Columns (0: agency_number, 1: responsibility_center_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")


In [178]:
len(df)

65000

## Sort and reorder


In [212]:
for i, col in enumerate(df.columns):
    print(f"{i}: '{col}',")

0: 'publication_date',
1: 'fiscal_year',
2: 'agency_number',
3: 'agency_name',
4: 'unit_appropriation_number',
5: 'unit_appropriation_name',
6: 'responsibility_center_code',
7: 'responsibility_center_name',
8: 'budget_code_number',
9: 'budget_code_name',
10: 'object_class_number',
11: 'object_class_name',
12: 'object_code',
13: 'object_code_name',
14: 'adopted_budget_amount',
15: 'current_modified_budget_amount',
16: 'financial_plan_amount',
17: 'adopted_budget_position',
18: 'current_modified_budget_position',
19: 'financial_plan_position',
20: 'adopted_budget_number_of_contracts',
21: 'current_modified_budget_number_of_contracts',
22: 'financial_plan_number_of_contracts',


In [ ]:
numeric_cols = [
    "adopted_budget_amount",
    "current_modified_budget_amount",
    "financial_plan_amount",
    
    "adopted_budget_position",
    "current_modified_budget_position",
    "financial_plan_position",
    
    "adopted_budget_number_of_contracts",
    "current_modified_budget_number_of_contracts",
    "financial_plan_number_of_contracts"
]

categorical_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    "object_code",
    "object_code_name",
    # "financial_plan_savings_flag",
    # "intra_city_purchase_code",
    # "personal_service_other_than_personal_service_indicator",
]

In [182]:
arr = list(df.columns)
for col in numeric_cols:
    arr.remove(col)
    
arr.remove("fiscal_year")
print(arr)

['publication_date', 'agency_number', 'agency_name', 'unit_appropriation_number', 'unit_appropriation_name', 'budget_code_number', 'budget_code_name', 'object_class_number', 'object_class_name', 'object_code', 'object_code_name', 'responsibility_center_code', 'responsibility_center_name', 'personal_service_other_than_personal_service_indicator', 'financial_plan_savings_flag', 'intra_city_purchase_code']


In [183]:
df[arr] = df[arr].fillna("(blank)")

In [184]:
for cat_col in categorical_cols:
    if cat_col in df.columns:
        df[cat_col] = df[cat_col].apply(lambda x: str.upper(x) if type(x) == str else x)

In [185]:
df = df.sort_values(
    [
        "agency_number",
        "unit_appropriation_number",
        "responsibility_center_code",
        "budget_code_number",
        "object_class_number",
        "object_code",
        "publication_date"
        ],
    ascending=[True, True, True, True, True, True, True]
    ).reset_index(drop=True)

df

,publication_date,fiscal_year,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,budget_code_number,budget_code_name,object_class_number,object_class_name,...,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount,adopted_budget_position,current_modified_budget_position,financial_plan_position,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts,intra_city_purchase_code
0,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0222,DEPUTY MAYOR FOR STRATEGIC POLICY,1,FULL TIME SALARIED,...,1158351,1158351,1158351,7,7,7,0,0,0,(blank)
1,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0222,DEPUTY MAYOR FOR STRATEGIC POLICY,1,FULL TIME SALARIED,...,1158351,1158351,1158351,7,7,7,0,0,0,(blank)
2,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0264,NYC SERVICE OFFICE,1,FULL TIME SALARIED,...,1137014,1165014,1165014,10,10,10,0,0,0,(blank)
3,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0264,NYC SERVICE OFFICE,1,FULL TIME SALARIED,...,1137014,1165014,1165014,10,10,10,0,0,0,(blank)
4,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0277,SENIOR ADVISOR TO THE MAYOR,1,FULL TIME SALARIED,...,3406413,3406413,3406413,24,24,24,0,0,0,(blank)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64995,20260217,2027,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,P010,FLEET REDUCTION,40,OTHER SERVICES AND CHARGES,...,0,0,-1060000000,0,0,0,0,0,0,(blank)
64996,20260512,2027,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,90,OTPS HOLDING CODES,...,0,0,-400000000,0,0,0,0,0,0,(blank)
64997,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P002,FINANCIAL PLAN SAVINGS,40,OTHER SERVICES AND CHARGES,...,0,0,37689973,0,0,0,0,0,0,(blank)
64998,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,P003,TIER IV RECS,40,OTHER SERVICES AND CHARGES,...,0,0,75000000,0,0,0,0,0,0,(blank)


In [186]:
# Reorder columns
df = df[
    [
    "publication_date", # 0: 
    "fiscal_year", # 1: 
    "agency_number", # 2: 
    "agency_name", # 3: 
    "unit_appropriation_number", # 4: 
    "unit_appropriation_name", # 5: 
    "responsibility_center_code", # 23: 
    "responsibility_center_name", # 24: 
    "budget_code_number", # 6: 
    "budget_code_name", # 7: 
    "object_class_number", # 8: 
    "object_class_name", # 9: 
    "object_code", # 10: 
    "object_code_name", # 11: 
    # "intra_city_purchase_code", # 25: 
    # "personal_service_other_than_personal_service_indicator", # 12: 
    # "financial_plan_savings_flag", # 13: 
    "adopted_budget_amount", # 14: 
    "current_modified_budget_amount", # 15: 
    "financial_plan_amount", # 16: 
    "adopted_budget_position", # 17: 
    "current_modified_budget_position", # 18: 
    "financial_plan_position", # 19: 
    "adopted_budget_number_of_contracts", # 20: 
    "current_modified_budget_number_of_contracts", # 21: 
    "financial_plan_number_of_contracts", # 22: 
    ]
]
df

,publication_date,fiscal_year,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,...,object_code_name,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount,adopted_budget_position,current_modified_budget_position,financial_plan_position,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts
0,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0222,DEPUTY MAYOR FOR STRATEGIC POLICY,...,FULL YEAR POSITIONS,1158351,1158351,1158351,7,7,7,0,0,0
1,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0222,DEPUTY MAYOR FOR STRATEGIC POLICY,...,FULL YEAR POSITIONS,1158351,1158351,1158351,7,7,7,0,0,0
2,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0264,NYC SERVICE OFFICE,...,FULL YEAR POSITIONS,1137014,1165014,1165014,10,10,10,0,0,0
3,20260512,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0264,NYC SERVICE OFFICE,...,FULL YEAR POSITIONS,1137014,1165014,1165014,10,10,10,0,0,0
4,20260217,2027,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,...,FULL YEAR POSITIONS,3406413,3406413,3406413,24,24,24,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64995,20260217,2027,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,(BLANK),(BLANK),P010,FLEET REDUCTION,...,OTHER EXPENSES - GENERAL,0,0,-1060000000,0,0,0,0,0,0
64996,20260512,2027,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P002,FINANCIAL PLAN SAVINGS,...,OTPS HOLDING CODE,0,0,-400000000,0,0,0,0,0,0
64997,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P002,FINANCIAL PLAN SAVINGS,...,OTHER EXPENSES - GENERAL,0,0,37689973,0,0,0,0,0,0
64998,20260217,2027,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P003,TIER IV RECS,...,OTHER EXPENSES - GENERAL,0,0,75000000,0,0,0,0,0,0


## Compare against 2027 master


In [187]:
master_2027_df = pd.read_excel("./2027/master_2027.xlsx", header=3)
master_2027_df_exec = master_2027_df[master_2027_df['publication_date'] == 20260512]

In [188]:
master_2027_df_exec

,publication_date,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,object_code,object_code_name,sum_of_adopted_budget_amount,sum_of_current_modified_budget_amount,sum_of_financial_plan_amount
0,20260512,2,MAYORALTY,20.0,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0229,Counsel to the Mayor,1.0,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1789231,1789231,1789231
1,20260512,2,MAYORALTY,20.0,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0230,Mayor's Judiciary Committee,1.0,FULL TIME SALARIED,001,FULL YEAR POSITIONS,88404,237404,88404
2,20260512,2,MAYORALTY,20.0,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0245,Comm to Combat Domestic Violence,1.0,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1870336,1870336,1870336
3,20260512,2,MAYORALTY,20.0,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,0250,Office of Immigrant Affairs,1.0,FULL TIME SALARIED,001,FULL YEAR POSITIONS,778962,778962,778962
4,20260512,2,MAYORALTY,20.0,OFFICE OF THE MAYOR-PS,0008,D/M FOR HUMAN SVC,0217,D/M FOR HEALTH & HUMAN SERVICES,1.0,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1348827,1348827,1348827
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32849,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2.0,OTHER THAN PERSONAL SERVICES,1,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,40.0,OTHER SERVICES AND CHARGES,42C,HEAT LIGHT & POWER,9027,9027,13075
32850,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2.0,OTHER THAN PERSONAL SERVICES,1,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,40.0,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,8688,7718,8688
32851,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2.0,OTHER THAN PERSONAL SERVICES,1,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,60.0,CONTRACTUAL SERVICES,600,CONTRACTUAL SERVICES GENERAL,0,9600,0
32852,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2.0,OTHER THAN PERSONAL SERVICES,(blank),(blank),P002,FINANCIAL PLAN SAVINGS,40.0,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,22363,97,-2637


In [189]:
master_2027_df_exec.columns

Index(['publication_date', 'agency_number', 'agency_name',
       'unit_appropriation_number', 'unit_appropriation_name',
       'responsibility_center_code', 'responsibility_center_name',
       'budget_code_number', 'budget_code_name', 'object_class_number',
       'object_class_name', 'object_code', 'object_code_name',
       'sum_of_adopted_budget_amount', 'sum_of_current_modified_budget_amount',
       'sum_of_financial_plan_amount'],
      dtype='str')

In [190]:
for cat_col in categorical_cols:
    if cat_col in master_2027_df_exec.columns:
        master_2027_df_exec[cat_col] = master_2027_df_exec[cat_col].apply(lambda x: str.upper(x) if type(x) == str else x)

In [191]:
df_pivot = df.groupby([
    'publication_date', 'agency_number', 'agency_name',
       'unit_appropriation_number', 'unit_appropriation_name',
       'responsibility_center_code', 'responsibility_center_name',
       'budget_code_number', 'budget_code_name', 'object_class_number',
       'object_class_name', 'object_code', 'object_code_name',
       ], dropna=False).sum().reset_index()

df_pivot = df_pivot[[
                     'publication_date', 'agency_number', 'agency_name',
       'unit_appropriation_number', 'unit_appropriation_name',
       'responsibility_center_code', 'responsibility_center_name',
       'budget_code_number', 'budget_code_name', 'object_class_number',
       'object_class_name', 'object_code', 'object_code_name',
       'adopted_budget_amount',
       'current_modified_budget_amount',
       'financial_plan_amount'
       ]]

df_pivot = df_pivot[df_pivot['publication_date'] == 20260512]

df_pivot

,publication_date,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,object_code,object_code_name,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount
31667,20260512,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0222,DEPUTY MAYOR FOR STRATEGIC POLICY,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1158351,1158351,1158351
31668,20260512,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0264,NYC SERVICE OFFICE,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1137014,1165014,1165014
31669,20260512,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,3406413,3406413,3406413
31670,20260512,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,3,UNSALARIED,031,UNSALARIED,85703,85703,85703
31671,20260512,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,5,AMOUNTS TO BE SCHEDULED,051,SALARY ADJUSTMENTS,9587,9587,9587
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64516,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,40,OTHER SERVICES AND CHARGES,42C,HEAT LIGHT & POWER,9027,9027,13075
64517,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,40,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,8688,7718,8688
64518,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,60,CONTRACTUAL SERVICES,600,CONTRACTUAL SERVICES GENERAL,0,9600,0
64519,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P002,FINANCIAL PLAN SAVINGS,40,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,22363,97,-2637


In [192]:
# 1. Left join with indicator=True
merged = df_pivot.merge(master_2027_df_exec, on=[
    'publication_date', 'agency_number', 'agency_name',
       'unit_appropriation_number', 'unit_appropriation_name',
       'responsibility_center_code', 'responsibility_center_name',
       'budget_code_number', 'budget_code_name', 'object_class_number',
       'object_class_name', 'object_code', 'object_code_name',
    ], how='outer', indicator=True)


merged

,publication_date,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,object_code,object_code_name,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount,sum_of_adopted_budget_amount,sum_of_current_modified_budget_amount,sum_of_financial_plan_amount,_merge
0,20260512,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0222,DEPUTY MAYOR FOR STRATEGIC POLICY,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1158351,1158351,1158351,1158351,1158351,1158351,both
1,20260512,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0264,NYC SERVICE OFFICE,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1137014,1165014,1165014,1137014,1165014,1165014,both
2,20260512,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,3406413,3406413,3406413,3406413,3406413,3406413,both
3,20260512,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,3,UNSALARIED,031,UNSALARIED,85703,85703,85703,85703,85703,85703,both
4,20260512,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,5,AMOUNTS TO BE SCHEDULED,051,SALARY ADJUSTMENTS,9587,9587,9587,9587,9587,9587,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32849,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,40,OTHER SERVICES AND CHARGES,42C,HEAT LIGHT & POWER,9027,9027,13075,9027,9027,13075,both
32850,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,40,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,8688,7718,8688,8688,7718,8688,both
32851,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,1.0,PUBLIC ADMINISTRATOR-RICHMOND,1000,DEPT OPER RICH COUNTY DIV,60,CONTRACTUAL SERVICES,600,CONTRACTUAL SERVICES GENERAL,0,9600,0,0,9600,0,both
32852,20260512,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P002,FINANCIAL PLAN SAVINGS,40,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,22363,97,-2637,22363,97,-2637,both


In [193]:
merged[(merged["adopted_budget_amount"] != merged["sum_of_adopted_budget_amount"]) | (merged["current_modified_budget_amount"] != merged["sum_of_current_modified_budget_amount"]) | (merged["financial_plan_amount"] != merged["sum_of_financial_plan_amount"])]

,publication_date,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,object_code,object_code_name,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount,sum_of_adopted_budget_amount,sum_of_current_modified_budget_amount,sum_of_financial_plan_amount,_merge


In [194]:
rows_not_in_both = merged[merged['_merge'] != 'both']
print(rows_not_in_both['_merge'].value_counts())

rows_not_in_both

_merge
left_only     0
right_only    0
both          0
Name: count, dtype: int64


,publication_date,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,object_code,object_code_name,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount,sum_of_adopted_budget_amount,sum_of_current_modified_budget_amount,sum_of_financial_plan_amount,_merge


## Track releases


In [195]:
pub_dates = df["publication_date"].sort_values(ascending=True).unique().tolist()[:]

num_pubs_this_year = len(pub_dates)

print(f"{num_pubs_this_year} pub_dates in FY {planned_fy}: {pub_dates}")

2 pub_dates in FY 2027: [20260217, 20260512]


In [196]:
df.to_excel(f"current_api_{planned_fy}.xlsx",sheet_name="Raw",index=False)

## Set up base DataFrame and map function


In [197]:
base_df = df[categorical_cols].drop_duplicates().reset_index(drop=True)
len(base_df)

33197

## Loop through each release


In [198]:
print(pub_dates)

print(planned_fy)

for i, pub_date in enumerate(pub_dates):
    release_name = RELEASE_NAMES[i]
    
    ith_release_df = df[df['publication_date'] == pub_date]
    
    print(i, release_name)

[20260217, 20260512]
2027
0 prelim
1 exec


## Collapse Releases


In [199]:
collapsed_df = base_df.copy()

new_numeric_cols = []


start_of_numerical_cols = len(categorical_cols)

# collapsed_df.head()

for i, pub_date in enumerate(pub_dates):
    print(f"[{i}/{num_pubs_this_year}] pub_date -- {pub_date} ({RELEASE_NAMES[i]}):")
    
    ith_release_df = df[df["publication_date"] == pub_date][categorical_cols + numeric_cols].copy()
    
    print(len(ith_release_df))
    
    # print(i)
    
    # print(ith_release_df.columns[len(categorical_cols):])
    
    if i <= 2:
        for col in ["financial_plan_amount", "financial_plan_position", "financial_plan_number_of_contracts"]:
            # print(col)
            ith_release_df = ith_release_df.rename(
                columns={
                    f"{col}":f"{col}_{RELEASE_NAMES[i]}",
                    }
            )
            # print()
            # print(ith_release_df.columns)
    
        if i < num_pubs_this_year - 1:
            ith_release_df = ith_release_df.drop(columns=[
                f"adopted_budget_amount",
                f"current_modified_budget_amount",
                
                f"adopted_budget_position",
                f"current_modified_budget_position",
                
                f"adopted_budget_number_of_contracts",
                f"current_modified_budget_number_of_contracts",
                ])
            
        print(f"Adding new numeric columns:{ith_release_df.columns[start_of_numerical_cols:]}\n")
        new_numeric_cols.extend(ith_release_df.columns[start_of_numerical_cols:])
    else:
        raise Exception(f"Bad indexing, i:{i} >= num_pubs_this_year or i:{i} < 0")
    
    # print(ith_release_df.columns)
    print()
    
    collapsed_df = collapsed_df.merge(right=ith_release_df,on=categorical_cols, how='left')
    
    # break

collapsed_df = collapsed_df.reset_index(drop=True)


print(len(base_df))
print(len(collapsed_df))


collapsed_df.columns

[0/2] pub_date -- 20260217 (prelim):
31881
Adding new numeric columns:Index(['financial_plan_amount_prelim', 'financial_plan_position_prelim',
       'financial_plan_number_of_contracts_prelim'],
      dtype='str')


[1/2] pub_date -- 20260512 (exec):
33119
Adding new numeric columns:Index(['adopted_budget_amount', 'current_modified_budget_amount',
       'financial_plan_amount_exec', 'adopted_budget_position',
       'current_modified_budget_position', 'financial_plan_position_exec',
       'adopted_budget_number_of_contracts',
       'current_modified_budget_number_of_contracts',
       'financial_plan_number_of_contracts_exec'],
      dtype='str')


33197
34822


Index(['agency_number', 'agency_name', 'unit_appropriation_number',
       'unit_appropriation_name', 'responsibility_center_code',
       'responsibility_center_name', 'budget_code_number', 'budget_code_name',
       'object_class_number', 'object_class_name', 'object_code',
       'object_code_name', 'financial_plan_amount_prelim',
       'financial_plan_position_prelim',
       'financial_plan_number_of_contracts_prelim', 'adopted_budget_amount',
       'current_modified_budget_amount', 'financial_plan_amount_exec',
       'adopted_budget_position', 'current_modified_budget_position',
       'financial_plan_position_exec', 'adopted_budget_number_of_contracts',
       'current_modified_budget_number_of_contracts',
       'financial_plan_number_of_contracts_exec'],
      dtype='str')

In [200]:
new_numeric_cols

['financial_plan_amount_prelim',
 'financial_plan_position_prelim',
 'financial_plan_number_of_contracts_prelim',
 'adopted_budget_amount',
 'current_modified_budget_amount',
 'financial_plan_amount_exec',
 'adopted_budget_position',
 'current_modified_budget_position',
 'financial_plan_position_exec',
 'adopted_budget_number_of_contracts',
 'current_modified_budget_number_of_contracts',
 'financial_plan_number_of_contracts_exec']

In [201]:
collapsed_df

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,...,financial_plan_number_of_contracts_prelim,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount_exec,adopted_budget_position,current_modified_budget_position,financial_plan_position_exec,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0222,DEPUTY MAYOR FOR STRATEGIC POLICY,1,FULL TIME SALARIED,...,0.0,1158351.0,1158351.0,1158351.0,7.0,7.0,7.0,0.0,0.0,0.0
1,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0264,NYC SERVICE OFFICE,1,FULL TIME SALARIED,...,0.0,1137014.0,1165014.0,1165014.0,10.0,10.0,10.0,0.0,0.0,0.0
2,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,1,FULL TIME SALARIED,...,0.0,3406413.0,3406413.0,3406413.0,24.0,24.0,24.0,0.0,0.0,0.0
3,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,3,UNSALARIED,...,0.0,85703.0,85703.0,85703.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,5,AMOUNTS TO BE SCHEDULED,...,0.0,9587.0,9587.0,9587.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34817,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,(BLANK),(BLANK),P010,FLEET REDUCTION,40,OTHER SERVICES AND CHARGES,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34818,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P002,FINANCIAL PLAN SAVINGS,90,OTPS HOLDING CODES,...,NaN,0.0,0.0,-400000000.0,0.0,0.0,0.0,0.0,0.0,0.0
34819,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P002,FINANCIAL PLAN SAVINGS,40,OTHER SERVICES AND CHARGES,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34820,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P003,TIER IV RECS,40,OTHER SERVICES AND CHARGES,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [202]:
collapsed_df[collapsed_df.duplicated(keep=False)]

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,...,financial_plan_number_of_contracts_prelim,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount_exec,adopted_budget_position,current_modified_budget_position,financial_plan_position_exec,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts_exec
728,3,BOARD OF ELECTIONS,2,OTHER THAN PERSONAL SERVICES,0002,DEPARTMENTAL OPERATIONS,0201,DEPARTMENTAL OPERATIONS,40,OTHER SERVICES AND CHARGES,...,0.0,0.0,16469.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
729,3,BOARD OF ELECTIONS,2,OTHER THAN PERSONAL SERVICES,0002,DEPARTMENTAL OPERATIONS,0201,DEPARTMENTAL OPERATIONS,40,OTHER SERVICES AND CHARGES,...,0.0,644.0,644.0,644.0,0.0,0.0,0.0,0.0,0.0,0.0
730,3,BOARD OF ELECTIONS,2,OTHER THAN PERSONAL SERVICES,0002,DEPARTMENTAL OPERATIONS,0201,DEPARTMENTAL OPERATIONS,40,OTHER SERVICES AND CHARGES,...,0.0,0.0,42609.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
731,3,BOARD OF ELECTIONS,2,OTHER THAN PERSONAL SERVICES,0002,DEPARTMENTAL OPERATIONS,0201,DEPARTMENTAL OPERATIONS,40,OTHER SERVICES AND CHARGES,...,0.0,0.0,16469.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
732,3,BOARD OF ELECTIONS,2,OTHER THAN PERSONAL SERVICES,0002,DEPARTMENTAL OPERATIONS,0201,DEPARTMENTAL OPERATIONS,40,OTHER SERVICES AND CHARGES,...,0.0,644.0,644.0,644.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33991,866,DEPT OF CONSUMER & WORKER PROTECTION,3,OTHER THAN PERSONAL SERVICE,18.0,BUDGET AND ADMINISTRATION,2601,FINANCE,40,OTHER SERVICES AND CHARGES,...,0.0,0.0,41184.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33992,866,DEPT OF CONSUMER & WORKER PROTECTION,3,OTHER THAN PERSONAL SERVICE,18.0,BUDGET AND ADMINISTRATION,2601,FINANCE,40,OTHER SERVICES AND CHARGES,...,0.0,0.0,3000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33996,866,DEPT OF CONSUMER & WORKER PROTECTION,3,OTHER THAN PERSONAL SERVICE,18.0,BUDGET AND ADMINISTRATION,2601,FINANCE,40,OTHER SERVICES AND CHARGES,...,0.0,102500.0,102500.0,102500.0,0.0,0.0,0.0,0.0,0.0,0.0
33997,866,DEPT OF CONSUMER & WORKER PROTECTION,3,OTHER THAN PERSONAL SERVICE,18.0,BUDGET AND ADMINISTRATION,2601,FINANCE,40,OTHER SERVICES AND CHARGES,...,0.0,0.0,41184.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Object Code Level


In [203]:
Object_Code_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    "object_code",
    "object_code_name",
    # "financial_plan_savings_flag",
]

main_cols = [
    'adopted_budget_amount',
    'current_modified_budget_amount',
    'financial_plan_amount_exec'
]

object_code_collapsed_df = collapsed_df.groupby(Object_Code_cols,dropna=False).sum().reset_index()

object_code_collapsed_df = object_code_collapsed_df[Object_Code_cols + new_numeric_cols]

object_code_collapsed_df[Object_Code_cols + main_cols]


,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,object_class_number,object_class_name,object_code,object_code_name,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0222,DEPUTY MAYOR FOR STRATEGIC POLICY,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1158351.0,1158351.0,1158351.0
1,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0264,NYC SERVICE OFFICE,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,1137014.0,1165014.0,1165014.0
2,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,1,FULL TIME SALARIED,001,FULL YEAR POSITIONS,3406413.0,3406413.0,3406413.0
3,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,3,UNSALARIED,031,UNSALARIED,85703.0,85703.0,85703.0
4,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,5,AMOUNTS TO BE SCHEDULED,051,SALARY ADJUSTMENTS,9587.0,9587.0,9587.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33192,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,(BLANK),(BLANK),P010,FLEET REDUCTION,40,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,0.0,0.0,0.0
33193,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P002,FINANCIAL PLAN SAVINGS,90,OTPS HOLDING CODES,999,OTPS HOLDING CODE,0.0,0.0,-400000000.0
33194,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P002,FINANCIAL PLAN SAVINGS,40,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,0.0,0.0,0.0
33195,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P003,TIER IV RECS,40,OTHER SERVICES AND CHARGES,499,OTHER EXPENSES - GENERAL,0.0,0.0,0.0


In [204]:
# object_code_collapsed_df[object_code_collapsed_df.duplicated(keep=False)]

## Budget Code Level


In [205]:
Budget_Code_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

main_cols = [
    'adopted_budget_amount',
    'current_modified_budget_amount',
    'financial_plan_amount_exec'
]

budget_code_collapsed_df = collapsed_df[Budget_Code_cols + new_numeric_cols].groupby(Budget_Code_cols,dropna=False).sum().reset_index()

# budget_code_collapsed_df = budget_code_collapsed_df[Budget_Code_cols + new_numeric_cols]

# budget_code_collapsed_df[Budget_Code_cols + main_cols]

budget_code_collapsed_df

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,budget_code_number,budget_code_name,financial_plan_amount_prelim,financial_plan_position_prelim,financial_plan_number_of_contracts_prelim,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount_exec,adopted_budget_position,current_modified_budget_position,financial_plan_position_exec,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0222,DEPUTY MAYOR FOR STRATEGIC POLICY,1.158351e+06,7.0,0.0,1158351.0,1158351.0,1158351.0,7.0,7.0,7.0,0.0,0.0,0.0
1,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0264,NYC SERVICE OFFICE,1.165014e+06,10.0,0.0,1137014.0,1165014.0,1165014.0,10.0,10.0,10.0,0.0,0.0,0.0
2,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0277,SENIOR ADVISOR TO THE MAYOR,3.501703e+06,24.0,0.0,3501703.0,3501703.0,3501703.0,24.0,24.0,24.0,0.0,0.0,0.0
3,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),0332,NYC TOURISM GRANT,0.000000e+00,0.0,0.0,81626.0,81626.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
4,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),P020,FINANCIAL PLAN SAVINGS,0.000000e+00,40.0,0.0,0.0,0.0,6286117.0,0.0,40.0,44.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8258,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,(BLANK),(BLANK),P010,FLEET REDUCTION,-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8259,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P002,FINANCIAL PLAN SAVINGS,0.000000e+00,0.0,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0,0.0,0.0,0.0
8260,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P002,FINANCIAL PLAN SAVINGS,3.768997e+07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8261,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),P003,TIER IV RECS,7.500000e+07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [206]:
# budget_code_collapsed_df[budget_code_collapsed_df.duplicated(keep=False)]

## Responsibility Center Level


In [207]:
RC_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

RC_collapsed_df = collapsed_df[RC_cols + new_numeric_cols].groupby(RC_cols,dropna=False).sum().reset_index()

# RC_collapsed_df = RC_collapsed_df[RC_cols + new_numeric_cols]

RC_collapsed_df
# RC_collapsed_df[RC_cols + main_cols]
# RC_collapsed_df[RC_collapsed_df['agency_name'] == "DEPARTMENT OF EDUCATION"][RC_cols + main_cols]

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,financial_plan_amount_prelim,financial_plan_position_prelim,financial_plan_number_of_contracts_prelim,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount_exec,adopted_budget_position,current_modified_budget_position,financial_plan_position_exec,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,(BLANK),(BLANK),5.825068e+06,81.0,0.0,5878694.0,5906694.0,12111185.0,42.0,82.0,85.0,0.0,0.0,0.0
1,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0006,COUNSEL TO THE MAYOR,4.526933e+06,26.0,0.0,4526933.0,4675933.0,4526933.0,26.0,26.0,26.0,0.0,0.0,0.0
2,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0008,D/M FOR HUMAN SVC,1.348827e+06,7.0,0.0,1348827.0,1348827.0,1348827.0,7.0,7.0,7.0,0.0,0.0,0.0
3,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0011,D/M FOR FINANCE AND ECO. DEV.,1.891294e+06,9.0,0.0,1891294.0,1891294.0,1891294.0,9.0,9.0,9.0,0.0,0.0,0.0
4,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,0028,D/M FOR OPERATIONS,1.248473e+06,7.0,0.0,1248473.0,1248473.0,1248473.0,7.0,7.0,7.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1918,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),-2.637000e+03,0.0,0.0,22363.0,97.0,-2637.0,0.0,0.0,0.0,0.0,0.0,0.0
1919,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,(BLANK),(BLANK),-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1920,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),0.000000e+00,0.0,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0,0.0,0.0,0.0
1921,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,(BLANK),(BLANK),1.126900e+08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [208]:
RC_collapsed_df[RC_collapsed_df.duplicated(keep=False)]

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,responsibility_center_code,responsibility_center_name,financial_plan_amount_prelim,financial_plan_position_prelim,financial_plan_number_of_contracts_prelim,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount_exec,adopted_budget_position,current_modified_budget_position,financial_plan_position_exec,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts_exec


## Unit of Appropriation Level


In [209]:
UoA_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    # "responsibility_center_code",
    # "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

UoA_collapsed_df = collapsed_df[UoA_cols + new_numeric_cols].groupby(UoA_cols,dropna=False).sum().reset_index()

# UoA_collapsed_df = UoA_collapsed_df[UoA_cols + new_numeric_cols]

UoA_collapsed_df
# UoA_collapsed_df[UoA_cols + main_cols]
# UoA_collapsed_df[UoA_collapsed_df['agency_name'] == "DEPARTMENT OF EDUCATION"][UoA_cols + main_cols]

,agency_number,agency_name,unit_appropriation_number,unit_appropriation_name,financial_plan_amount_prelim,financial_plan_position_prelim,financial_plan_number_of_contracts_prelim,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount_exec,adopted_budget_position,current_modified_budget_position,financial_plan_position_exec,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts_exec
0,2,MAYORALTY,20,OFFICE OF THE MAYOR-PS,4.046508e+07,318.0,0.0,40515979.0,40316729.0,46751202.0,319.0,319.0,322.0,0.0,0.0,0.0
1,2,MAYORALTY,21,OFFICE OF THE MAYOR-OTPS,4.489982e+06,0.0,14.0,4237982.0,4459520.0,5031799.0,0.0,0.0,0.0,14.0,18.0,14.0
2,2,MAYORALTY,40,OFFICE OF MGMT AND BUDGET-PS,5.606719e+07,467.0,0.0,55440649.0,54732488.0,55900321.0,455.0,452.0,467.0,0.0,0.0,0.0
3,2,MAYORALTY,41,OFFICE OF MGMT AND BUDGET-OTPS,1.135147e+07,0.0,23.0,11308977.0,11554610.0,11669072.0,0.0,0.0,0.0,23.0,23.0,23.0
4,2,MAYORALTY,61,OFF OF LABOR RELATIONS-PS,1.811062e+07,188.0,0.0,16885354.0,17866722.0,18597617.0,173.0,193.0,193.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
722,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,3.105700e+04,0.0,0.0,56057.0,56057.0,35105.0,0.0,0.0,0.0,0.0,1.0,0.0
723,99C,CITYWIDE SAVINGS INITIATIVES,2,CITYWIDE SAVINGS - OTPS,-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
724,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,0.000000e+00,0.0,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0,0.0,0.0,0.0
725,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,1.126900e+08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [210]:
# UoA_collapsed_df[UoA_collapsed_df.duplicated(keep=False)]

## Agency Level


In [211]:
Agency_cols = [
    "agency_number",
    "agency_name",
    # "unit_appropriation_number",
    # "unit_appropriation_name",
    # "responsibility_center_code",
    # "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

Agency_collapsed_df = collapsed_df[Agency_cols + new_numeric_cols].groupby(Agency_cols,dropna=False).sum().reset_index()

# Agency_collapsed_df = Agency_collapsed_df[Agency_cols + new_numeric_cols]

# Agency_collapsed_df[Agency_cols + main_cols]
Agency_collapsed_df

,agency_number,agency_name,financial_plan_amount_prelim,financial_plan_position_prelim,financial_plan_number_of_contracts_prelim,adopted_budget_amount,current_modified_budget_amount,financial_plan_amount_exec,adopted_budget_position,current_modified_budget_position,financial_plan_position_exec,adopted_budget_number_of_contracts,current_modified_budget_number_of_contracts,financial_plan_number_of_contracts_exec
0,2,MAYORALTY,1.881819e+08,1311.0,65.0,196148751.0,197371734.0,199360409.0,1289.0,1306.0,1332.0,65.0,77.0,65.0
1,3,BOARD OF ELECTIONS,1.468794e+08,517.0,37.0,146879436.0,216784225.0,206976967.0,517.0,517.0,517.0,37.0,33.0,37.0
2,4,CAMPAIGN FINANCE BOARD,1.347705e+07,87.0,27.0,109481237.0,126865022.0,104093091.0,258.0,258.0,258.0,9.0,11.0,10.0
3,8,OFFICE OF THE ACTUARY,7.617044e+06,42.0,9.0,7614099.0,7677099.0,8483139.0,42.0,42.0,42.0,9.0,10.0,9.0
4,10,BOROUGH PRESIDENT - MANHATTAN,5.586632e+06,56.0,0.0,6017382.0,6194656.0,6079019.0,56.0,56.0,56.0,0.0,9.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,6.555800e+05,5.0,0.0,673602.0,673602.0,657628.0,5.0,5.0,5.0,0.0,1.0,0.0
142,99C,CITYWIDE SAVINGS INITIATIVES,-1.060000e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
143,99D,PRIOR YEAR PAYABLES,0.000000e+00,0.0,0.0,0.0,0.0,-400000000.0,0.0,0.0,0.0,0.0,0.0,0.0
144,99P,ENERGY ADJUSTMENT,1.126900e+08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
